# An iterative perceptron, second attempt

**Status:** ✅ Runs end to end — both update branches previously ran the same line, so it could not learn

**Demonstrates:** Linear classification. The original also swapped argument order between definition and call sites.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


In [0]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

*1*. define step *function*

In [ ]:

def step(z):
    """Apply the Heaviside step function.

    Args:
        z: Scalar or array of net inputs.

    Returns:
        numpy.ndarray or bool: ``1`` where ``z > 0``, otherwise ``0``.
    """
    return z > 0


2. define error

In [ ]:

def count_misclassifications(X, y, w, b):
    """Count how many samples the current hyperplane gets wrong.

    BUGFIX: the original was `error(y, x, w, b)`, which took four arguments
    but was called with three from `derr`, raising `TypeError`. It also
    returned the number of *correct* predictions, so the training loop's
    stopping condition `abs(e) > epsilon` was true forever.

    Args:
        X: Feature matrix of shape ``(n_samples, n_features)``.
        y: Boolean labels of shape ``(n_samples,)``.
        w: Weight vector of shape ``(n_features,)``.
        b: Scalar bias.

    Returns:
        int: Number of misclassified samples.
    """
    predictions = (X @ w + b) > 0
    return int(np.sum(predictions != y))


In [ ]:

def predict_row(x, w, b):
    """Predict the class of a single sample.

    BUGFIX: the original signature was `y_pred(w, x, b=1)` but every call site
    passed the arguments the other way round, so the sample was multiplied by
    the weights and vice versa. The order is now ``(x, w, b)`` everywhere.

    Args:
        x: Feature vector of shape ``(n_features,)``.
        w: Weight vector of shape ``(n_features,)``.
        b: Scalar bias.

    Returns:
        bool: ``True`` if the sample is classified as positive.
    """
    return bool(step(np.dot(x, w) + b))





calc y_hat:

defrential of error by w -> x+w0

In [ ]:

def weight_correction(x, y_true, y_pred):
    """Return the per-sample gradient direction of the perceptron update.

    BUGFIX: the original `derr(x)` closed over undefined globals `y` and `w`,
    so it could never run.

    Args:
        x: Feature vector of shape ``(n_features,)``.
        y_true: Integer label, ``0`` or ``1``.
        y_pred: Predicted label, ``0`` or ``1``.

    Returns:
        numpy.ndarray: ``x`` when the prediction is wrong, otherwise zeros. The
        sign of the correction is carried by ``y_true - y_pred`` in
        :func:`train_perceptron`.
    """
    return x if y_true != y_pred else np.zeros_like(x)


updating w: 

In [ ]:

def update_weights(x, y_true, y_pred, w, b, learning_rate):
    """Apply one perceptron update to the weights and bias.

    BUGFIX: the original `if` and `elif` branches of the training loop both
    executed the identical line `w -= x @ learn_rate`, so the correction sign
    never changed and the rule could not learn anything. The sign now comes
    from ``y_true - y_pred`` in a single branch.

    Args:
        x: Feature vector of shape ``(n_features,)``.
        y_true: True label, ``0`` or ``1``.
        y_pred: Predicted label, ``0`` or ``1``.
        w: Weight vector of shape ``(n_features,)``; not modified in place.
        b: Scalar bias.
        learning_rate: Step size for the update.

    Returns:
        tuple: ``(w_new, b_new)`` as a copy of the updated parameters.
    """
    error = y_true - y_pred
    w_new = w + learning_rate * error * x
    b_new = b + learning_rate * error
    return w_new, b_new


inetelize and fit preceptron 

In [0]:
# def train_weights(train, l_rate, n_epoch):
# 	weights = [0.0 for i in range(len(train[0]))]
# 	for epoch in range(n_epoch):
# 		sum_error = 0.0
# 		for row in train:
# 			prediction = predict(row, weights)
# 			error = row[-1] - prediction
# 			sum_error += error**2
# 			weights[0] = weights[0] + l_rate * error
# 			for i in range(len(row)-1):
# 				weights[i + 1] = weights[i + 1] + l_rate * error * row[i]
# 		print('>epoch=%d, lrate=%.3f, error=%.3f' % (epoch, l_rate, sum_error))
# 	return weights

In [ ]:

def train_perceptron(X, y, learning_rate=0.1, max_epochs=100, random_state=0):
    """Train a perceptron with the online update rule.

    BUGFIX, three parts:

    1. The original loop had no epoch cap and its stopping condition never
       became false, so a non-separable dataset would spin forever.
    2. ``wt = w`` then ``wt[-1] = b`` aliased the weight array, so writing the
       bias into the last slot silently overwrote a real weight each pass.
       The bias is now a separate scalar.
    3. The error count was measured once before the loop and never refreshed.

    Args:
        X: Feature matrix of shape ``(n_samples, n_features)``.
        y: Binary labels of shape ``(n_samples,)`` with values in ``{0, 1}``.
        learning_rate: Step size for each update.
        max_epochs: Hard cap on passes over the data.
        random_state: Seed for the per-epoch shuffle.

    Returns:
        tuple: ``(w, b, history)`` where ``w`` has shape ``(n_features,)``,
        ``b`` is the learned bias, and ``history`` lists the misclassification
        count at the end of each epoch.

    Example:
        >>> import numpy as np
        >>> X = np.array([[1.0, 1.0], [1.0, 0.5], [-1.0, -1.0], [-0.5, -1.0]])
        >>> y = np.array([1, 1, 0, 0])
        >>> w, b, hist = train_perceptron(X, y, random_state=0)
        >>> hist[-1]
        0
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y).ravel().astype(int)
    rng = np.random.default_rng(random_state)

    w = np.zeros(X.shape[1], dtype=float)
    b = 0.0
    history = []

    for _ in range(max_epochs):
        order = rng.permutation(X.shape[0])
        mistakes = 0
        for i in order:
            y_pred = int(step(np.dot(X[i], w) + b))
            if y_pred != y[i]:
                w, b = update_weights(X[i], y[i], y_pred, w, b, learning_rate)
                mistakes += 1
        history.append(mistakes)
        if mistakes == 0:
            break
    return w, b, history


data :

In [ ]:

# Synthetic separable problem: two features, labels from a fixed linear
# boundary, with a margin band around the boundary removed.
FEATURE_COUNT = 2
SAMPLES = 1000
rng = np.random.default_rng(0)

X = rng.random((SAMPLES, FEATURE_COUNT))
boundary = np.array([-0.01, -0.40904551, 0.33732037])   # includes the bias term
X_bias = np.hstack((np.ones(SAMPLES)[:, np.newaxis], X))
signed_distance = X_bias @ boundary

# Keep only samples comfortably clear of the boundary, so the problem is
# separable and the perceptron is guaranteed to converge.
margin = np.abs(signed_distance) > 0.07
X = X[margin]
y = (signed_distance[margin] > 0).astype(int)
print("samples kept:", X.shape[0], "| class balance:", y.sum(), "vs", len(y) - y.sum())


In [ ]:

w, b, history = train_perceptron(X, y, learning_rate=0.1, max_epochs=100, random_state=0)
print("learned weights:", w, " bias:", round(b, 4))
print("misclassifications per epoch:", history)


In [ ]:

train_acc = np.mean(((X @ w + b) > 0).astype(int) == y)
print(f"X.shape={X.shape}  y.shape={y.shape}  training accuracy={train_acc:.4f}")


In [ ]:

# BUGFIX: the original cell called `w.append(b)`. numpy.ndarray has no
# .append method, so this raised AttributeError - and it would also have
# mutated the learned weights had it worked. The bias is a separate scalar,
# so the augmented design matrix is all that is needed.
X_aug = np.hstack([X, np.ones((X.shape[0], 1))])
print("augmented shape:", X_aug.shape, "| bias column ones:", int(X_aug[:, -1].sum()))


In [ ]:

# BUGFIX: the original was `w.append(-1, np.array([b]))`, which is doubly
# wrong - ndarray has no .append, and the real signature takes one argument.
# The learned parameters are already correct, so this cell just shows them.
print(f"w = {w}")
print(f"b = {b}")


In [ ]:

# Verify the learned boundary directly against the generator's boundary.
X_aug_aug = np.hstack([X, np.ones((X.shape[0], 1))])
learned_distance = X_aug_aug @ np.append(w, b)
print("sign agreement with generator:",
      f"{np.mean(np.sign(learned_distance) == np.sign(signed_distance[margin])):.4f}")


In [ ]:

# BUGFIX: this cell ran twice and printed a result with no meaning. The dot
# product below is the actual forward pass for a single sample.
x = np.array([1.0, 2.0, 3.0])
w3 = np.array([0.5, -1.0, 2.0])
print("x @ w  =", x @ w3, " (shape", (x @ w3).shape, ")")
